### RAG Pipelines-  to Vector DB Pipline

In [28]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
# from langchain_community.document_loader import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []

    # find all pdf files recursively
    pdf_files = list(Path(pdf_directory).glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"  X Error: {e}")
    print(f"\nTotal Documents loaded: {len(all_documents)}")
    return all_documents

# process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")
print(all_pdf_documents[:2])  # preview first 2 docs


Found 2 PDF files to process

Processing: Computer Science IX.pdf
  Loaded 168 pages

Processing: Computer Science X.pdf


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)


chunks = splitter.split_documents(all_pdf_documents)
print(f"Created {len(chunks)} chunks")


Created 1155 chunks


In [ ]:
def split_documents(documents, chunk_size=100, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    return text_splitter.split_documents(documents)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=500, chunk_overlap=50):
    """Split documents into smaller chunks for better RAG performance"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = splitter.split_documents(documents)

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [ ]:
chunks = split_documents(all_pdf_documents)
print(chunks)


Example chunk:
Content: Serial No.
Date of Publication Edition Quantity Price
Publisher Code No. 
All rights are reserved with the SINDH TEXTBOOK, BOARD, JAMSHORO.
Prepared by for SINDH ASSOCIATION FOR ACADEMIC QUALITY (AFAQ...
Metadata: {'producer': 'Wondershare PDFelement', 'creator': 'Wondershare PDFelement', 'creationdate': '2020-08-19T13:39:17+05:00', 'title': '', 'author': 'Muhammad Faheem DPA', 'moddate': 'D\x00:\x002\x000\x002\x000\x000\x008\x001\x009\x001\x003', 'source': '..\\data\\pdf\\Computer Science IX.pdf', 'total_pages': 168, 'page': 0, 'page_label': '1', 'source_file': 'Computer Science IX.pdf', 'file_type': 'pdf'}
[Document(metadata={'producer': 'Wondershare PDFelement', 'creator': 'Wondershare PDFelement', 'creationdate': '2020-08-19T13:39:17+05:00', 'title': '', 'author': 'Muhammad Faheem DPA', 'moddate': 'D\x00:\x002\x000\x002\x000\x000\x008\x001\x009\x001\x003', 'source': '..\\data\\pdf\\Computer Science IX.pdf', 'total_pages': 168, 'page': 0, 'page_label': '1', 

In [ ]:
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 1155


In [ ]:
for i in range(2):  # check first 2 overlaps
    print(f"\n--- Chunk {i+1} end ---")
    print(chunks[i].page_content[-100:])   # last 100 characters
    print(f"--- Chunk {i+2} start ---")
    print(chunks[i+1].page_content[:100])  # first 100 characters



--- Chunk 1 end ---
ards of Intermediate and secondary 
Education, Karachi, Hyderbad, Sukkur, Larkana and Mirpurkhas for
--- Chunk 2 start ---
Secondary School Certificate Examination in the Province of Sindh.
Approved by the Education and Lit

--- Chunk 2 end ---
de999
THE TEXTBOOK OFTHE TEXTBOOK OFTHE TEXTBOOK OF
COMPUTERCOMPUTER
SCIENCESCIENCE
COMPUTER
SCIENCE
--- Chunk 3 start ---
COMPUTERCOMPUTER
SCIENCESCIENCE
COMPUTER
SCIENCE
SINDH TEXTBOOK BOARD, JAMSHOROSINDH TEXTBOOK BOARD,


### Embedding and vectgorStoreDb

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# embeddgin
class EmbeddingManager:
    """Handle documents embedding generation using SentenceTransformer"""

    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):

        """
        Initialize the embeddnig manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model() # next function -> laod the model

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimeension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts
        Args:
            texts: List of text strings embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embedding with shape: {embeddings.shape}")
        return embeddings
    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_embedding_dimension()

## initilize teh embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4069.38it/s]


Model loaded successfully. Embedding dimeension: 384


### VectorStore

In [ ]:
from importlib import metadata
import os

#explain VectorStore class: 

class VectorStore:
    """Manages document embedding in a chromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_sore"):

        """
        Initialize the vector store
        Args:
            collection_name : Name of hte chromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """initiallize ChromaDB  client and collection"""
        try:
            #Create persistent ChromaDb cient
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
                                                                   metadata={"description": "Collection of PDF document embeddings for RAG"})
            print(f"Vector store initialized with collection: {self.collection_name} at {self.persist_directory}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Args:
            documents: List of document metadata (dicts)
            embeddings: numpy array of embeddings with shape (len(documents), embedding_dim)
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        # prepare data for ChromaDb
        ids = []
        metadatas = []
        document_texts = []
        embedding_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # prepare metadata and text
            metadata = dict(doc.metadata)  # make a copy of the metadata dictionary
            metadata['doc_index'] = i  # add index to metadata
            metadata['content_length'] = len(doc.page_content)  # add content length to metadata
            metadatas.append(metadata)

            # prepare document text and embedding
            document_texts.append(doc.page_content)
            #Embedding 
            embedding_list.append(embedding.tolist())  # convert numpy array to list for ChromaDB


        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embedding_list
            )
            print(f"Sucessfully added {len(documents)} documents to the vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:  
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store


Vector store initialized with collection: pdf_documents at ../data/vector_sore
Existing documents in collection: 0


In [ ]:
chunks

[Document(metadata={'producer': 'Wondershare PDFelement', 'creator': 'Wondershare PDFelement', 'creationdate': '2020-08-19T13:39:17+05:00', 'title': '', 'author': 'Muhammad Faheem DPA', 'moddate': 'D\x00:\x002\x000\x002\x000\x000\x008\x001\x009\x001\x003', 'source': '..\\data\\pdf\\Computer Science IX.pdf', 'total_pages': 168, 'page': 0, 'page_label': '1', 'source_file': 'Computer Science IX.pdf', 'file_type': 'pdf'}, page_content='Serial No.\nDate of Publication Edition Quantity Price\nPublisher Code No. \nAll rights are reserved with the SINDH TEXTBOOK, BOARD, JAMSHORO.\nPrepared by for SINDH ASSOCIATION FOR ACADEMIC QUALITY (AFAQ) \nTEXT BOOK BOARD JAMSHORO. \nReviewed by Provincial  Review Committee Directorate of Curriculum \nAssessment and Research Sindh Jamshoro (DCAR).\nPrescribed as a Textbook by the Boards of Intermediate and secondary \nEducation, Karachi, Hyderbad, Sukkur, Larkana and Mirpurkhas for'),
 Document(metadata={'producer': 'Wondershare PDFelement', 'creator': 'Wo

In [ ]:
### convert the text to embeddings and add to vector store
texts = [doc.page_content for doc in chunks]
texts

['Serial No.\nDate of Publication Edition Quantity Price\nPublisher Code No. \nAll rights are reserved with the SINDH TEXTBOOK, BOARD, JAMSHORO.\nPrepared by for SINDH ASSOCIATION FOR ACADEMIC QUALITY (AFAQ) \nTEXT BOOK BOARD JAMSHORO. \nReviewed by Provincial  Review Committee Directorate of Curriculum \nAssessment and Research Sindh Jamshoro (DCAR).\nPrescribed as a Textbook by the Boards of Intermediate and secondary \nEducation, Karachi, Hyderbad, Sukkur, Larkana and Mirpurkhas for',
 'Secondary School Certificate Examination in the Province of Sindh.\nApproved by the Education and Literacy Department, Government of Sindh.\nNo. SED/HCW/181/2018 Dated 22-7-2020 for the province of Sindh. \nTHE TEXTBOOK OF       FOR CLASS 9 THE TEXTBOOK OF       FOR CLASS 9               COMPUTERCOMPUTER THE TEXTBOOK OF       FOR CLASS 9        COMPUTER\nFor GradeFor GradeFor Grade999\nTHE TEXTBOOK OFTHE TEXTBOOK OFTHE TEXTBOOK OF\nCOMPUTERCOMPUTER\nSCIENCESCIENCE\nCOMPUTER\nSCIENCE',
 'COMPUTERCOMPU

In [ ]:
### convert the text to embeddings and add to vector store
texts = [doc.page_content for doc in chunks]

## Generate the embeddings for the chunks
embeddings = embedding_manager.generate_embedding(texts)

embeddings.shape

Generating embeddings for 1155 texts...


Batches: 100%|██████████| 37/37 [00:43<00:00,  1.18s/it]

Generated Embedding with shape: (1155, 384)


(1155, 384)

In [ ]:
embeddings

array([[ 0.00683276,  0.03580142, -0.0874725 , ..., -0.1382181 ,
        -0.02647847, -0.00216393],
       [-0.09895977,  0.05512593, -0.03098732, ...,  0.02261972,
        -0.00811203,  0.03177945],
       [-0.03693648, -0.04051021, -0.01927053, ..., -0.05069447,
        -0.07626586,  0.02358929],
       ...,
       [-0.0593568 ,  0.0049113 , -0.12702557, ..., -0.07588289,
        -0.04362324,  0.03932618],
       [-0.0593568 ,  0.0049113 , -0.12702557, ..., -0.07588289,
        -0.04362324,  0.03932618],
       [-0.05935682,  0.00491129, -0.12702559, ..., -0.07588294,
        -0.04362322,  0.03932622]], shape=(1155, 384), dtype=float32)

In [ ]:
### convert the text to embeddings and add to vector store
texts = [doc.page_content for doc in chunks]

## Generate the embeddings for the chunks
embeddings = embedding_manager.generate_embedding(texts)

#store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 1155 texts...


Batches: 100%|██████████| 37/37 [00:41<00:00,  1.12s/it]


Generated Embedding with shape: (1155, 384)
Sucessfully added 1155 documents to the vector store
Total documents in collection: 1155


### Retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever
        Args:
            vector_store: Instance of VectorStore for document retrieval
            embedding_manager: Instance of EmbeddingManager for query embedding
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a given query
        Args:
            query: User query string
            top_k: Number of top documents to retrieve
            score_threshold: Minimum similarity score to consider a document relevant
        Returns:
            List of dictionaries containing document metadata and similarity scores
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embedding([query])[0]  # get the first embedding query to embedding

        #search in vector store
        try:
            ## hit the vector store to retrieve relevant documents
            results = self.vector_store.collection.query(
                        query_embeddings=[query_embedding.tolist()],
                        n_results=top_k
                    )
            #process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                embeddings = results['embeddings'][0]
                ids = results['ids'][0]

                for i, (doc_id, documents, metadata, distance) in enumerate(zip(ids, documents, metadatas, embeddings)):
                    # Calculate cosine similarity
                    similarity_score = 1 - distance  # Assuming distance is in [0, 1], where 1 is identical and 0 is completely different


                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': documents,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}")
            else:
                print("No documents retrieved from the vector store.")
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []
    rag_retriever = RAGRetriever(vector_store, embedding_manager)


NameError: name 'RAGRetriever' is not defined